In [ ]:
#!/usr/bin/env python3
"""
Integrated Order‑Flow Imbalance (OFI)
====================================

Compute the **Integrated OFI** time series exactly as defined in
*“Cross‑Impact of Order‑Flow Imbalance”*, Section 2.1.3–2.1.4.

Steps implemented
-----------------
1.  Read the **normalised multi‑level OFI matrix** from `multi_level_ofi.csv`.
2.  Centre the matrix (subtract column means) and apply PCA **exactly**:
      • Singular‑Value Decomposition (SVD) on the centred matrix.  
      • Take the first right‑singular vector ``w₁``.  
3.  Normalise ``w₁`` so that ``‖w₁‖₁ = 1`` (Eq. (4)).  
4.  Project every timestamp’s OFI vector onto ``w₁`` to obtain a scalar
    **integrated OFI** value.
5.  Save the single‑column result to `integrated_level_ofi.csv`.

**No LASSO, sparsity tricks, or regression** are used—this is *pure PCA*.

---------------------------------------------------------------------------
Author  : ChatGPT — Market‑Microstructure Feature Engineer
Version : 1.0
---------------------------------------------------------------------------

"""

from __future__ import annotations

from pathlib import Path
from typing import List

import numpy as np
import pandas as pd


# ── CONFIGURE INPUT / OUTPUT PATHS ────────────────────────────────────────
MULTI_OFI_CSV: Path = Path("multi_level_ofi.csv")        # input (normalised)
OUT_CSV:       Path = Path("integrated_level_ofi.csv")   # output (scalar series)

TIMESTAMP_COL: str = "timestamp"                         # timestamp column name
OFI_PREFIX:    str = "ofi_level_"                        # column prefix to select


# ── CORE FUNCTION ─────────────────────────────────────────────────────────


def compute_integrated_ofi(df_multi: pd.DataFrame) -> pd.Series:
    """
    Compute the **Integrated OFI** (Eq. 4) from a multi‑level OFI matrix.

    Parameters
    ----------
    df_multi : pandas.DataFrame
        Must be indexed by datetime (UTC) and contain columns named
        ``ofi_level_00 … ofi_level_{M-1}`` representing *normalised* OFI at
        each book level.

    Returns
    -------
    pandas.Series
        A single column named ``ofi_integrated`` indexed by the same timestamps.
    """
    # 1. Select OFI columns in order
    ofi_cols: List[str] = [c for c in df_multi.columns if c.startswith(OFI_PREFIX)]
    if not ofi_cols:
        raise ValueError("No columns starting with 'ofi_level_' found.")

    X = df_multi[ofi_cols].values.astype(float)

    # 2. Centre the matrix (subtract column means)
    col_means = X.mean(axis=0, keepdims=True)
    X_centered = X - col_means

    # 3. PCA via SVD (first right‑singular vector)
    #    X_centered = U Σ Vᵀ  →  first PC = V[0]
    _, _, Vt = np.linalg.svd(X_centered, full_matrices=False)
    w1 = Vt[0]                     # leading PC (shape = (M,))

    # 4. L¹‑normalise (Eq. 4):  Σ|w₁ᵢ| = 1
    l1_norm = np.sum(np.abs(w1))
    if l1_norm == 0:
        raise ValueError("Leading PC has zero L¹‑norm; cannot normalise.")
    w1_norm = w1 / l1_norm

    # 5. Project each timestamp’s OFI vector onto w₁
    integrated_values = X.dot(w1_norm)

    integrated_series = pd.Series(
        integrated_values,
        index=df_multi.index,
        name="ofi_integrated",
        dtype=float,
    )

    return integrated_series


# ── MAIN SCRIPT (example usage) ───────────────────────────────────────────
if __name__ == "__main__":
    # 1. Load multi‑level OFI matrix
    df_multi = pd.read_csv(
        MULTI_OFI_CSV,
        parse_dates=[TIMESTAMP_COL],
        index_col=TIMESTAMP_COL,
    ).sort_index()                       # ensure time‑ascending order

    # 2. Compute Integrated OFI
    ofi_integrated = compute_integrated_ofi(df_multi)

    # 3. Save to CSV & show preview
    ofi_integrated.to_csv(OUT_CSV, header=True)
    print(f"Integrated OFI written to {OUT_CSV}  ({len(ofi_integrated)} rows)")
    print(ofi_integrated.head())


FileNotFoundError: [Errno 2] No such file or directory: 'multi_level_ofi.csv'